In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import fftconvolve

from ipywidgets import (
    FloatSlider,
    RadioButtons,
    HTML,
    HTMLMath,
    VBox,
    HBox,
    Layout
)

from IPython.display import display

# ============================================================
# LAPLACE EQUATION IN THE UPPER HALF-PLANE
#
# phi_xx + phi_yy = 0
#
# y > 0
#
# phi(x,0) = f(x)
#
# Poisson integral:
#
# phi(x,y) =
# y/pi integral f(xi) /
# [y^2 + (x-xi)^2] dxi
# ============================================================

plt.ioff()

display(HTML("""
<style>

.container { width:98% !important; max-width:none !important; }

.output_area,
.output_subarea,
.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.output_scroll {
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
    box-shadow:none !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow:visible !important;
    resize:none !important;
}

.poisson-title {
    font-family:Arial;
    font-size:20px;
    font-weight:bold;
    color:#6f3fa0;
}

.poisson-label {
    font-family:Arial;
    font-size:14px;
    font-weight:bold;
}

.poisson-value {
    font-family:Arial;
    font-size:14px;
    font-weight:bold;
    color:#0b3d91;
}

.poisson-radio .widget-radio-box {
    display:flex !important;
    flex-direction:row !important;
    flex-wrap:nowrap !important;
    gap:18px !important;
}

.poisson-radio > label {
    display:none !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1180px;
    font-family:Arial;
    font-size:15px;
    line-height:1.48;
    margin-bottom:10px;
">

<div class="poisson-title" style="margin-bottom:8px;">
Laplace Equation and the Poisson Kernel
</div>

<div style="margin-bottom:5px;">
The Laplace equation φₓₓ+φᵧᵧ=0 is considered in the upper
half-plane y&gt;0 with boundary condition φ(x,0)=f(x).
</div>

<div style="margin-bottom:5px;">
Fourier transformation with respect to x converts the partial
differential equation into an ordinary differential equation
in y. After inverse transformation the solution becomes the
Poisson integral.
</div>

<div style="margin-bottom:5px;">
As y increases, the Poisson kernel averages the boundary data
over an increasingly wide neighborhood. The harmonic extension
therefore becomes progressively smoother.
</div>

<div>
<b>This notebook:</b> displays the boundary function, the harmonic
solution at a selected height y, and the complete solution in the
upper half-plane.
</div>

</div>
""")

poisson_math = HTMLMath(
    value=(
        r'\('
        r'\varphi(x,y)='
        r'\frac{y}{\pi}'
        r'\displaystyle\int_{-\infty}^{\infty}'
        r'\frac{f(\xi)}{y^2+(x-\xi)^2}'
        r'd\xi'
        r'\)'
    )
)

# ============================================================
# GRID
# ============================================================

NX = 700
XMAX = 6.0

x = np.linspace(
    -XMAX,
    XMAX,
    NX
)

dx = (
    x[1] - x[0]
)

YMAX = 3.0
NY = 90

y_grid = np.linspace(
    0.05,
    YMAX,
    NY
)

# ============================================================
# BOUNDARY FUNCTION
# ============================================================

boundary_selector = RadioButtons(
    options=[
        ('Rectangular pulse', 'rectangle'),
        ('Gaussian', 'gaussian'),
        ('Double pulse', 'double')
    ],
    value='rectangle',
    description='',
    layout=Layout(width='550px')
)

boundary_selector.add_class(
    'poisson-radio'
)

# ============================================================
# HEIGHT SLIDER
# ============================================================

slider_style = {
    'description_width': '0px'
}

y_slider = FloatSlider(
    min=0.05,
    max=YMAX,
    step=0.05,
    value=0.50,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=Layout(width='260px')
)

y_value = HTML(
    '<div class="poisson-value">0.50</div>',
    layout=Layout(width='65px')
)

height_row = HBox(
    [
        HTML(
            '<div class="poisson-label">Height y:</div>',
            layout=Layout(width='100px', min_width='100px')
        ),
        y_slider,
        y_value
    ],
    layout=Layout(
        width='450px',
        height='40px',
        align_items='center'
    )
)

controls = VBox(
    [
        HTML('<div class="poisson-title" style="margin-bottom:7px;">Boundary Data</div>'),
        boundary_selector,
        height_row
    ],
    layout=Layout(
        width='500px',
        padding='10px 14px',
        border='1px solid #d2c2df'
    )
)

current_math = HTMLMath()

current_panel = VBox(
    [
        HTML('<div class="poisson-title" style="margin-bottom:7px;">Current Slice</div>'),
        poisson_math,
        current_math
    ],
    layout=Layout(
        width='620px',
        padding='10px 14px',
        border='1px solid #d2c2df'
    )
)

top_row = HBox(
    [controls, current_panel],
    layout=Layout(width='1140px', gap='15px', align_items='stretch')
)

# ============================================================
# BOUNDARY FUNCTIONS
# ============================================================

def boundary_function(mode):
    if mode == 'rectangle':
        return (
            np.abs(x) <= 1.0
        ).astype(float)

    if mode == 'gaussian':
        return np.exp(
            -x**2
        )

    return (
        (
            np.abs(x - 1.5) <= 0.55
        ).astype(float)
        +
        0.70
        *
        (
            np.abs(x + 1.5) <= 0.55
        ).astype(float)
    )

# ============================================================
# POISSON KERNEL
# ============================================================

def poisson_kernel(y):
    return (
        y
        /
        (
            np.pi
            *
            (y**2 + x**2)
        )
    )

# ============================================================
# HARMONIC EXTENSION
# ============================================================

def harmonic_slice(f_values, y):
    kernel = poisson_kernel(y)

    return (
        fftconvolve(
            f_values,
            kernel,
            mode='same'
        )
        *
        dx
    )

def harmonic_field(f_values):
    field = np.zeros(
        (
            NY,
            NX
        )
    )

    for j, y_value_local in enumerate(y_grid):
        field[j, :] = harmonic_slice(
            f_values,
            y_value_local
        )

    return field

# ============================================================
# INITIAL DATA
# ============================================================

f_initial = boundary_function(
    boundary_selector.value
)

field_initial = harmonic_field(
    f_initial
)

slice_initial = harmonic_slice(
    f_initial,
    y_slider.value
)

# ============================================================
# FIGURE 1 — BOUNDARY AND SLICE
# ============================================================

fig_slice, ax_slice = plt.subplots(
    figsize=(6.3, 4.8)
)

fig_slice.canvas.header_visible = False
fig_slice.canvas.footer_visible = False
fig_slice.canvas.toolbar_visible = False

fig_slice.canvas.layout = Layout(
    width='630px',
    height='480px'
)

ax_slice.set_title(
    'Boundary Data and Harmonic Extension',
    fontsize=14,
    fontweight='bold',
    color='#6f3fa0'
)

ax_slice.set_xlabel('x')
ax_slice.set_ylabel('Value')
ax_slice.set_xlim(-XMAX, XMAX)
ax_slice.set_ylim(-0.10, 1.20)
ax_slice.grid(True, linestyle=':', alpha=0.40)

boundary_line, = ax_slice.plot(
    x,
    f_initial,
    linewidth=1.6,
    linestyle='--',
    label='f(x)'
)

slice_line, = ax_slice.plot(
    x,
    slice_initial,
    linewidth=2.2,
    label='φ(x,y)'
)

ax_slice.legend(
    loc='upper right',
    fontsize=9
)

fig_slice.subplots_adjust(
    left=0.10,
    right=0.97,
    top=0.90,
    bottom=0.13
)

# ============================================================
# FIGURE 2 — UPPER HALF-PLANE HEATMAP
# ============================================================

fig_field, ax_field = plt.subplots(
    figsize=(5.0, 4.8)
)

fig_field.canvas.header_visible = False
fig_field.canvas.footer_visible = False
fig_field.canvas.toolbar_visible = False

fig_field.canvas.layout = Layout(
    width='500px',
    height='480px'
)

ax_field.set_title(
    'Harmonic Solution in the Upper Half-Plane',
    fontsize=12,
    fontweight='bold',
    color='#0b3d91'
)

ax_field.set_xlabel('x')
ax_field.set_ylabel('y')

field_image = ax_field.imshow(
    field_initial,
    extent=[
        -XMAX,
        XMAX,
        y_grid[0],
        y_grid[-1]
    ],
    origin='lower',
    aspect='auto',
    vmin=0.0,
    vmax=1.0,
    interpolation='bilinear'
)

selected_height_line = ax_field.axhline(
    y_slider.value,
    linestyle='--',
    linewidth=1.3
)

fig_field.colorbar(
    field_image,
    ax=ax_field,
    fraction=0.046,
    pad=0.04
)

fig_field.subplots_adjust(
    left=0.13,
    right=0.90,
    top=0.90,
    bottom=0.13
)

figures_row = HBox(
    [
        fig_slice.canvas,
        fig_field.canvas
    ],
    layout=Layout(
        width='1140px',
        gap='10px',
        align_items='flex-start'
    )
)

# ============================================================
# UPDATE SELECTED HEIGHT
# ============================================================

current_boundary = f_initial.copy()
current_field = field_initial.copy()

def update_height(change=None):
    y_current = y_slider.value

    solution = harmonic_slice(
        current_boundary,
        y_current
    )

    slice_line.set_ydata(
        solution
    )

    selected_height_line.set_ydata(
        [y_current, y_current]
    )

    y_value.value = (
        f'<div class="poisson-value">{y_current:.2f}</div>'
    )

    current_math.value = (
        r'\('
        r'y='
        + f'{y_current:.4f}'
        + r'\)'
    )

    fig_slice.canvas.draw_idle()
    fig_field.canvas.draw_idle()

# ============================================================
# UPDATE BOUNDARY FUNCTION
# ============================================================

def update_boundary(change=None):
    global current_boundary, current_field

    current_boundary = boundary_function(
        boundary_selector.value
    )

    current_field = harmonic_field(
        current_boundary
    )

    boundary_line.set_ydata(
        current_boundary
    )

    field_image.set_data(
        current_field
    )

    update_height()

    fig_slice.canvas.draw_idle()
    fig_field.canvas.draw_idle()

boundary_selector.observe(
    update_boundary,
    names='value'
)

y_slider.observe(
    update_height,
    names='value'
)

update_height()

display(
    VBox(
        [
            documentation,
            top_row,
            figures_row
        ],
        layout=Layout(
            width='1180px',
            gap='10px'
        )
    )
)